# Week 2: Logistics Data Collection, Cleaning and Preprocessing

This notebook reproduces the preprocessing workflow documented in the Word report.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW_FILE = ROOT / 'data' / 'raw' / 'logistics_data.csv'
df = pd.read_csv(RAW_FILE)
df.head()

## 1. Initial data-quality inspection

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
df.describe(include='all')

## 2. Remove duplicates and handle missing values

In [ ]:
num_cols = ['Quantity', 'Shipping_Cost', 'Delivery_Time']
cat_cols = ['Transport_Mode', 'Destination_Region', 'Status']

df = df.drop_duplicates().copy()

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(df.isnull().sum())

## 3. Detect outliers with IQR

Potential outliers are flagged for business investigation rather than automatically deleted.

In [ ]:
def iqr_bounds(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in num_cols:
    lower, upper = iqr_bounds(df, col)
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: {len(outliers)} potential outliers; bounds=({lower:.2f}, {upper:.2f})')

## 4. Min-Max normalization

In [ ]:
scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df.to_csv(ROOT / 'data' / 'processed' / 'cleaned_logistics_data.csv', index=False)
df.head()

## 5. Final validation

In [ ]:
assert df.isnull().sum().sum() == 0
assert df.duplicated().sum() == 0
print('Preprocessing validation passed.')
print('Final shape:', df.shape)